<img src="https://upload.wikimedia.org/wikipedia/commons/8/89/TheNorthFace_logo.svg" alt="LOGO THE NORTH FACE" width="40%" />

# The North Face E-commerce – Booster les Ventes en Ligne avec le NLP et l'Apprentissage Non Supervisé

**Type de Projet :** Machine Learning Non Supervisé (NLP + Clustering + Modélisation de Sujets)  
**Jeu de données :** Catalogue produits The North Face (~500 références)  
**Compétences Jedha :** Bloc 3 – Analyse prédictive / Traitement de données textuelles avec Scikit-Learn  

---

## Contexte Métier

The North Face est une marque américaine iconique fondée en 1968. Le département marketing souhaite exploiter le Machine Learning pour :

1. **Regrouper les produits similaires** par clustering, afin de mieux comprendre le catalogue
2. **Construire un système de recommandation** basé sur ces clusters ("Vous aimerez aussi...")
3. **Extraire les sujets latents** des descriptions produits via la modélisation de sujets (LSA)

Il s'agit d'un projet **NLP + Apprentissage Non Supervisé** pur — il n'y a pas de variable cible étiquetée. La performance est évaluée par le **Score de Silhouette** (clustering) et la **Variance Expliquée** (LSA).

## 0. Configuration de l'Environnement

In [ ]:
%pip install pandas plotly scikit-learn spacy wordcloud matplotlib nbformat ipywidgets

In [ ]:
# Téléchargement du modèle spaCy anglais pour le prétraitement NLP (lemmatisation, suppression des mots vides)
import subprocess
subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

In [ ]:
# ── Bibliothèques principales ────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# ── NLP ──────────────────────────────────────────────────────────────────────
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Pipeline Scikit-Learn ────────────────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# ── Clustering ───────────────────────────────────────────────────────────────
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

# ── Modélisation de Sujets (LSA) ─────────────────────────────────────────────
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings('ignore')
print('Toutes les bibliothèques chargées avec succès !')

---
## 1. Chargement des Données & Analyse Exploratoire (EDA)

### 1.1 Chargement du Jeu de Données

In [ ]:
# Chargement du catalogue produits The North Face
# Source des données : https://www.kaggle.com/cclark/product-item-data
df = pd.read_csv('sample-data.csv')
print(f'Dimensions du jeu de données : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head(5)

### 1.2 Informations Générales

In [ ]:
print('=== Informations sur le Jeu de Données ===')
print(f'Nombre de produits : {len(df)}')
print(f'Valeurs manquantes :\n{df.isnull().sum()}')
print(f'Lignes dupliquées : {df.duplicated().sum()}')
df.info()

### 1.3 EDA – Distribution de la Longueur des Textes

**Pourquoi cette visualisation ?**  
Comprendre la distribution de la longueur des descriptions permet d'évaluer la richesse des informations disponibles pour les modèles NLP. Des descriptions très courtes créeront des vecteurs TF-IDF creux, ce qui peut nuire à la qualité du clustering.

In [ ]:
# Calcul de la longueur brute du texte (nombre de caractères) avant tout nettoyage
df['text_length'] = df['description'].astype(str).apply(len)
df['word_count']  = df['description'].astype(str).apply(lambda x: len(x.split()))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Nombre de Caractères par Description',
                                    'Nombre de Mots par Description'))

fig.add_trace(go.Histogram(x=df['text_length'], nbinsx=30,
                            marker_color='#0057A8', name='Nb caractères',
                            opacity=0.85),
              row=1, col=1)

fig.add_trace(go.Histogram(x=df['word_count'], nbinsx=30,
                            marker_color='#E8341B', name='Nb mots',
                            opacity=0.85),
              row=1, col=2)

fig.update_layout(title='Distribution de la Longueur des Descriptions',
                  template='plotly_white', height=400, showlegend=False)
fig.show()

print(f'Moyenne de caractères par description : {df.text_length.mean():.0f}')
print(f'Moyenne de mots par description       : {df.word_count.mean():.0f}')

### 1.4 EDA – Top 30 des Mots les Plus Fréquents (Bruts)

**Pourquoi cette visualisation ?**  
Identifier les mots les plus fréquents dans le texte brut révèle : (a) quels mots vides dominent et doivent être supprimés, (b) quels mots récurrents spécifiques aux produits existent. Cela valide le choix d'appliquer la lemmatisation et la suppression des mots vides avant le TF-IDF.

In [ ]:
from collections import Counter

# Analyse rapide de la fréquence brute des mots (avant prétraitement)
all_words = ' '.join(df['description'].astype(str).str.lower()).split()
word_freq = Counter(all_words)
top30 = pd.DataFrame(word_freq.most_common(30), columns=['mot', 'compte'])

fig = px.bar(top30, x='compte', y='mot', orientation='h',
             title='Top 30 des Mots les Plus Fréquents (Texte Brut)',
             color='compte', color_continuous_scale='Blues',
             template='plotly_white', height=550)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print('\nObservation : Les mots fonctionnels ("and", "the", "a", "for") dominent.')
print('=> Cela confirme la nécessité de supprimer les mots vides et de lemmatiser.')

### 1.5 EDA – Mots-Clés Produits (Après Nettoyage Simple)

**Pourquoi cette visualisation ?**  
Nous appliquons un nettoyage simple (suppression des balises HTML) et filtrons uniquement les tokens alphabétiques pour voir quelles catégories de produits émergent naturellement dans le vocabulaire. Cela aide à anticiper le nombre des clusters significatifs.

In [ ]:
# Suppression simple des balises HTML pour extraire le vocabulaire produit
def basic_clean(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))  # Suppression des balises HTML
    text = re.sub(r'[^a-zA-Z ]', ' ', text)     # Conserver uniquement les caractères alphabétiques
    return text.lower().strip()

# Mots vides génériques à filtrer
generic_stops = {'the', 'and', 'a', 'of', 'for', 'to', 'with', 'in', 'is',
                 'it', 'an', 'our', 'are', 'br', 'li', 'details', 'fabric',
                 'weight', 'made', 'b', 'ul', 'has', 'at', 'on', 'or', 'that',
                 'from', 'this', 'its', 'by', 'be', 'all', 'have', 'as', 'not',
                 'when', 'can', 'you', 'also', 'which', 'but', 'we', 'your',
                 'their', 'they', 'so', 'more', 'into', 'no', 'will', 'one'}

cleaned_words = []
for desc in df['description'].astype(str):
    cleaned = basic_clean(desc)
    words = [w for w in cleaned.split() if w not in generic_stops and len(w) > 3]
    cleaned_words.extend(words)

top_keywords = pd.DataFrame(Counter(cleaned_words).most_common(25),
                             columns=['mot_cle', 'compte'])

fig = px.bar(top_keywords, x='mot_cle', y='compte',
             title='Top 25 Mots-Clés Significatifs (Après Nettoyage Simple)',
             color='compte', color_continuous_scale='Oranges',
             template='plotly_white', height=400)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

print('Observation : Les mots-clés matériaux dominent (polyester, nylon, coton, recyclé).')
print('=> Les produits se regroupent probablement par type de matériau et activité.')

---
## 2. Prétraitement du Texte & Feature Engineering

### Justification de la Stratégie

Pour le NLP sur les descriptions produits, nous suivons ce pipeline :

| Étape | Outil | Justification |
|-------|-------|---------------|
| Suppression HTML | `re.sub()` | Les descriptions contiennent du HTML brut (`<br>`, `<li>`) |
| Mise en minuscules + tokenisation | spaCy | Prétraitement NLP standard |
| Suppression des mots vides | spaCy | Supprime les mots parasites sans sens sémantique |
| Lemmatisation | spaCy | Réduit les formes des mots à la base (ex. `wicking` → `wick`) |
| Vectorisation TF-IDF | Scikit-Learn | Pondère les termes rares mais informatifs par rapport aux termes communs |

> ⚠️ **Pas de fuite de données (Data Leakage) :** Le TF-IDF est ajusté sur l'ensemble du corpus (pas besoin de train/test split pour l'apprentissage non supervisé). Le `Pipeline` garantit que toutes les étapes sont chaînées et reproductibles.

In [ ]:
# Chargement du modèle spaCy anglais
nlp = spacy.load('en_core_web_sm')
print('Modèle spaCy chargé :', nlp.meta['name'])

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Pipeline complet de prétraitement NLP :
    1. Suppression des balises HTML (présentes dans les données brutes)
    2. Conservation des caractères alphabétiques uniquement
    3. Tokenisation, suppression des mots vides, lemmatisation avec spaCy
    """
    # Étape 1 : Suppression des balises HTML et caractères spéciaux
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    
    # Étape 2 : Application de spaCy (tokenisation + lemmatisation + suppression mots vides)
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop       # Supprimer les mots vides
        and not token.is_punct     # Supprimer la ponctuation
        and token.is_alpha         # Conserver uniquement les tokens alphabétiques
        and len(token.lemma_) > 2  # Supprimer les tokens très courts
    ]
    return ' '.join(tokens)


# Application du prétraitement à toutes les descriptions
print('Prétraitement des descriptions en cours...')
df['clean_description'] = df['description'].apply(preprocess_text)
print('Terminé !')

# Comparaison avant/après
print('\n=== Avant Prétraitement ===')
print(df['description'].iloc[0][:300])
print('\n=== Après Prétraitement ===')
print(df['clean_description'].iloc[0])

In [ ]:
# ── Pipeline Scikit-Learn : FunctionTransformer + TF-IDF ────────────────────
# Le FunctionTransformer encapsule notre fonction preprocess_text personnalisée
# pour l'intégrer proprement dans un Pipeline Scikit-Learn (sans fuite de données).

vectorizer = TfidfVectorizer(
    max_features=3000,  # Limite le vocabulaire aux 3000 termes les plus informatifs
    min_df=2,           # Ignore les termes apparaissant dans moins de 2 documents
    max_df=0.85,        # Ignore les termes apparaissant dans plus de 85% des documents
    ngram_range=(1, 2)  # Utilise des unigrammes ET des bigrammes pour plus de contexte
)

tfidf_pipeline = Pipeline([
    ('preprocesseur', FunctionTransformer(lambda X: [preprocess_text(x) for x in X],
                                           validate=False)),
    ('tfidf', vectorizer)
])

# Ajustement et transformation du pipeline sur les descriptions brutes
tfidf_matrix = tfidf_pipeline.fit_transform(df['description'])

print(f'Dimensions de la matrice TF-IDF : {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} produits × {tfidf_matrix.shape[1]} caractéristiques (termes)')
print(f'\nCreusité de la matrice : {100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.1f}%')

In [ ]:
# Visualisation des termes TF-IDF les plus importants (score TF-IDF moyen sur tous les documents)
feature_names = tfidf_pipeline.named_steps['tfidf'].get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf_df = pd.DataFrame({'terme': feature_names, 'tfidf_moyen': avg_tfidf})\
                  .sort_values('tfidf_moyen', ascending=False).head(20)

fig = px.bar(top_tfidf_df, x='tfidf_moyen', y='terme', orientation='h',
             title='Top 20 Termes par Score TF-IDF Moyen',
             color='tfidf_moyen', color_continuous_scale='Viridis',
             template='plotly_white', height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## 3. Partie 1 – Clustering des Produits avec DBSCAN

### Justification du Choix du Modèle : Pourquoi DBSCAN ?

| Modèle | Avantages | Inconvénients | Verdict |
|--------|-----------|---------------|---------|
| **K-Means** | Rapide, simple | Nécessite de connaître k ; utilise la distance euclidienne (mauvais pour le texte) | ❌ Inadapté aux TF-IDF creux |
| **DBSCAN** | Pas besoin de spécifier k ; gère les valeurs aberrantes ; fonctionne avec la distance cosinus | Sensible à `eps`/`min_samples` | ✅ Meilleur choix pour le texte |
| **Agglomératif** | Crée une hiérarchie | Très lent sur 500 éléments ; gourmand en mémoire | ❌ Trop lent |

**Paramètre clé :** `metric='cosine'` — pour le texte, la similarité cosinus mesure l'angle entre les vecteurs TF-IDF (direction = similarité thématique), pas leur magnitude (longueur du document).

In [ ]:
# ── Recherche d'hyperparamètres pour DBSCAN ──────────────────────────────────
# Nous testons différentes valeurs d'eps et suivons : nombre de clusters,
# % de valeurs aberrantes, et Score de Silhouette (si au moins 2 clusters trouvés).

results = []
eps_values = np.arange(0.05, 0.55, 0.05)

for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=2, metric='cosine')
    labels = dbscan.fit_predict(tfidf_matrix)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_noise = n_noise / len(labels) * 100
    
    sil = None
    if n_clusters >= 2:
        # Le score de silhouette est calculé uniquement sur les points non aberrants
        mask = labels != -1
        if mask.sum() > 1:
            sil = silhouette_score(tfidf_matrix[mask], labels[mask],
                                   metric='cosine', sample_size=min(500, mask.sum()))
    
    results.append({'eps': round(eps, 2), 'n_clusters': n_clusters,
                    'n_bruit': n_noise, 'pct_bruit': round(pct_noise, 1),
                    'silhouette': round(sil, 4) if sil else None})

results_df = pd.DataFrame(results)
print(results_df.to_string())

In [ ]:
# Visualisation des résultats du balayage de paramètres
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Nombre de Clusters en fonction d\'eps',
                                    'Pourcentage Aberrants en fonction d\'eps'))

fig.add_trace(go.Scatter(x=results_df['eps'], y=results_df['n_clusters'],
                         mode='lines+markers', marker_color='#0057A8',
                         name='# Clusters'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=results_df['eps'], y=results_df['pct_bruit'],
                         mode='lines+markers', marker_color='#E8341B',
                         name='% Aberrants'),
              row=1, col=2)

fig.update_layout(title='Réglage des Hyperparamètres DBSCAN (balayage eps)',
                  template='plotly_white', height=380)
fig.show()

print('Le point idéal est là où l\'on obtient 10-20 clusters avec un taux d\'aberrants acceptable.')

In [ ]:
# Sélection de l'eps optimal selon : 10-20 clusters et taux d'aberrants acceptable
valid = results_df[(results_df['n_clusters'] >= 8) & (results_df['pct_bruit'] <= 40)]
if len(valid) > 0 and valid['silhouette'].notna().any():
    best_row = valid.dropna(subset=['silhouette']).sort_values('silhouette', ascending=False).iloc[0]
    BEST_EPS = best_row['eps']
else:
    BEST_EPS = 0.3  # Valeur par défaut

print(f'eps sélectionné = {BEST_EPS}')

In [ ]:
# ── Entraînement du modèle DBSCAN final ──────────────────────────────────────
dbscan_final = DBSCAN(eps=BEST_EPS, min_samples=2, metric='cosine')
cluster_labels = dbscan_final.fit_predict(tfidf_matrix)

df['cluster'] = cluster_labels

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_outliers = (cluster_labels == -1).sum()

print(f'Modèle final : eps={BEST_EPS}, min_samples=2')
print(f'Nombre de clusters trouvés   : {n_clusters}')
print(f'Nombre de valeurs aberrantes : {n_outliers} ({n_outliers/len(df)*100:.1f}%)')

# Calcul du Score de Silhouette pour les points non aberrants
mask = cluster_labels != -1
if mask.sum() > 1 and n_clusters >= 2:
    sil_score = silhouette_score(tfidf_matrix[mask], cluster_labels[mask],
                                 metric='cosine')
    print(f'Score de Silhouette (cosinus) : {sil_score:.4f}')
    print('  → Interprétation : 0 = aléatoire, 1 = séparation parfaite')

In [ ]:
# Visualisation de la distribution des tailles de clusters
cluster_counts = df[df['cluster'] != -1].groupby('cluster').size().reset_index(name='compte')
cluster_counts['cluster'] = cluster_counts['cluster'].astype(str)

fig = px.bar(cluster_counts, x='cluster', y='compte',
             title='Nombre de Produits par Cluster DBSCAN',
             color='compte', color_continuous_scale='Blues',
             labels={'cluster': 'ID du Cluster', 'compte': 'Nombre de Produits'},
             template='plotly_white', height=380)
fig.show()

print(f"\nAberrants (cluster = -1) : {(df['cluster'] == -1).sum()} produits")

### 3.1 Nuages de Mots (WordClouds) par Cluster

Les nuages de mots affichent les termes les plus fréquents et distinctifs dans chaque cluster, aidant à **interpréter les clusters** en les associant à des catégories de produits (équipement de pêche, vêtements de running, sous-vêtements techniques, etc.).

In [ ]:
# Génération d'un nuage de mots pour chaque cluster (max 12 clusters affichés)
unique_clusters = sorted([c for c in df['cluster'].unique() if c != -1])
n_display = min(len(unique_clusters), 12)

cols = 3
rows_wc = int(np.ceil(n_display / cols))
fig_wc, axes = plt.subplots(rows_wc, cols, figsize=(18, 5 * rows_wc))
axes = axes.flatten()

for i, cluster_id in enumerate(unique_clusters[:n_display]):
    cluster_texts = df[df['cluster'] == cluster_id]['clean_description'].str.cat(sep=' ')
    if cluster_texts.strip():
        wc = WordCloud(width=400, height=250,
                       background_color='white',
                       colormap='Blues',
                       max_words=40).generate(cluster_texts)
        axes[i].imshow(wc, interpolation='bilinear')
        n_items = (df['cluster'] == cluster_id).sum()
        axes[i].set_title(f'Cluster {cluster_id}  ({n_items} produits)', fontsize=12, fontweight='bold')
    axes[i].axis('off')

# Masquer les axes inutilisés
for j in range(n_display, len(axes)):
    axes[j].axis('off')

plt.suptitle('Nuages de Mots – Termes Caractéristiques par Cluster', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('wordclouds_clusters.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Partie 2 – Système de Recommandation

Nous utilisons les attributions de clusters de la Partie 1 pour construire un **système de recommandation basé sur le contenu**. La logique est simple :
- Les produits du même cluster ont des descriptions similaires → ils sont "similaires"
- Au sein d'un cluster, on classe par similarité cosinus par rapport à l'article demandé pour retourner les **5 produits les plus similaires**

In [ ]:
# Conversion de la matrice TF-IDF creuse en tableau dense pour les calculs de similarité
tfidf_dense = tfidf_matrix.toarray()

def find_similar_items(item_id: int, top_n: int = 5) -> pd.DataFrame:
    """
    Étant donné un item_id de produit, retourne les top_n produits les plus similaires
    en se basant sur :
      1) L'appartenance au même cluster DBSCAN
      2) La similarité cosinus la plus élevée par rapport à l'article demandé (dans le cluster)

    Args:
        item_id (int): La valeur 'id' du produit d'intérêt.
        top_n   (int): Nombre de recommandations à retourner (défaut 5).

    Returns:
        pd.DataFrame: DataFrame avec les ids et descriptions des produits recommandés.
    """
    # Trouver l'index de ligne correspondant à item_id
    idx_matches = df.index[df['id'] == item_id].tolist()
    if not idx_matches:
        return pd.DataFrame({'message': [f'Produit id={item_id} introuvable.']})
    idx = idx_matches[0]

    # Récupérer le cluster de l'article demandé
    query_cluster = df.loc[idx, 'cluster']

    if query_cluster == -1:
        # Produit aberrant : recours à la similarité cosinus globale
        print(f'Le produit {item_id} est un aberrant DBSCAN – utilisation de la similarité globale.')
        cluster_indices = df.index.tolist()
    else:
        # Obtenir tous les indices de produits du même cluster (excluant l'article demandé)
        cluster_indices = df.index[df['cluster'] == query_cluster].tolist()

    cluster_indices = [i for i in cluster_indices if i != idx]

    if not cluster_indices:
        return pd.DataFrame({'message': ['Aucun produit similaire trouvé dans le même cluster.']})

    # Calcul des distances cosinus entre l'article demandé et tous les membres du cluster
    query_vec    = tfidf_dense[idx].reshape(1, -1)
    cluster_vecs = tfidf_dense[cluster_indices]
    distances    = cosine_distances(query_vec, cluster_vecs).flatten()

    # Tri par distance croissante (= plus similaire en premier)
    top_indices = np.argsort(distances)[:top_n]
    recommended_df_indices = [cluster_indices[i] for i in top_indices]

    result = df.loc[recommended_df_indices, ['id', 'description']].copy()
    result['similarite'] = 1 - distances[top_indices]  # Conversion distance → similarité
    result['description'] = result['description'].str[:120] + '...'
    return result.reset_index(drop=True)


# ── Démonstration : recommandations pour le produit id=1 ────────────────────
query_id = 1
query_name = df[df['id'] == query_id]['description'].values[0][:80]
print(f'Produit demandé (id={query_id}) : {query_name}...')
print(f'\nCluster attribué : {df[df["id"] == query_id]["cluster"].values[0]}')
print('\n=== Top 5 Recommandations ===')
find_similar_items(query_id)

In [ ]:
# ── Démonstration interactive (pour Jupyter) ─────────────────────────────────
print('='*60)
print('DÉMONSTRATION INTERACTIVE DU SYSTÈME DE RECOMMANDATION')
print('='*60)
print(f'IDs de produits disponibles : 1 à {df["id"].max()}')
print('\nExemples de produits :')
for i in [1, 5, 12, 30, 50]:
    print(f'  id={i}: {df[df["id"]==i]["description"].values[0][:70]}...')

# Utilisez input() pour un usage interactif ; commentez pour l'exécution automatique
# user_input = input('\nEntrez un ID de produit pour obtenir des recommandations : ')
# chosen_id = int(user_input)
# print(find_similar_items(chosen_id))

# Démonstration statique pour l'exécution automatique :
chosen_id = 5
print(f'\n--- Démo auto pour le produit id={chosen_id} ---')
print(find_similar_items(chosen_id))

---
## 5. Partie 3 – Modélisation de Sujets avec LSA (TruncatedSVD)

> ⚠️ **Cette partie est indépendante des Parties 1 & 2.**

### Justification du Choix du Modèle : Pourquoi LSA (TruncatedSVD) ?

| Modèle | Point Fort | Faiblesse |
|--------|------------|-----------|
| **LSA (TruncatedSVD)** | Rapide, algèbre linéaire, sujets interprétables | Ne gère pas parfaitement la polysémie |
| **LDA** | Probabiliste, mélange de sujets natif | Beaucoup plus lent ; nécessite des comptages entiers (pas TF-IDF) |
| **NMF** | Non-négatif = parties additives, interprétable | Convergence plus lente que LSA |

**LSA est le point de départ recommandé** pour ce type de projet de catalogue e-commerce grâce à sa rapidité et son interprétabilité.

In [ ]:
# ── LSA : TruncatedSVD ───────────────────────────────────────────────────────
# Nous testons n_components = 15 (cherchant 10-20 sujets significatifs)
N_TOPICS = 15

lsa = TruncatedSVD(n_components=N_TOPICS, random_state=42)
topic_encoded = lsa.fit_transform(tfidf_matrix)
topic_encoded_df = pd.DataFrame(topic_encoded,
                                 columns=[f'Sujet_{i}' for i in range(N_TOPICS)])

explained_variance = lsa.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f'LSA avec {N_TOPICS} sujets')
print(f'Variance totale expliquée : {cumulative_variance[-1]*100:.1f}%')
print(f'\nVariance expliquée par sujet :')
for i, ev in enumerate(explained_variance):
    print(f'  Sujet {i:2d} : {ev*100:.2f}%  (cumulée : {cumulative_variance[i]*100:.1f}%)')

In [ ]:
# Visualisation de la variance expliquée (sert d'importance des caractéristiques pour LSA)
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Variance Expliquée par Sujet',
                                    'Variance Expliquée Cumulée'))

fig.add_trace(go.Bar(x=[f'S{i}' for i in range(N_TOPICS)],
                     y=explained_variance * 100,
                     marker_color='#0057A8', name='Par sujet'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=[f'S{i}' for i in range(N_TOPICS)],
                          y=cumulative_variance * 100,
                          mode='lines+markers', marker_color='#E8341B',
                          name='Cumulée'),
              row=1, col=2)

fig.update_layout(title='LSA – Variance Expliquée (Évaluation du Modèle)',
                  template='plotly_white', height=380, showlegend=False)
fig.update_yaxes(title_text='Variance Expliquée (%)')
fig.show()

### 5.1 Importance des Caractéristiques : Top Mots par Sujet

Dans LSA, l'**importance d'un terme** pour un sujet donné est donnée par sa **charge** (valeur de la composante) dans la décomposition SVD. Une charge absolue élevée = terme très représentatif de ce sujet.

In [ ]:
# Extraction des 10 mots par sujet (charges positives = plus associés)
feature_names_lsa = tfidf_pipeline.named_steps['tfidf'].get_feature_names_out()
n_top_words = 10

print(f'=== Top {n_top_words} mots par Sujet LSA (Importance des Caractéristiques) ===')
topic_keywords = {}
for topic_idx, component in enumerate(lsa.components_):
    top_word_indices = component.argsort()[-n_top_words:][::-1]
    top_words = [feature_names_lsa[i] for i in top_word_indices]
    top_scores = [component[i] for i in top_word_indices]
    topic_keywords[topic_idx] = top_words
    print(f'  Sujet {topic_idx:2d} : {", ".join(top_words)}')

In [ ]:
# Attribution du sujet dominant (score le plus élevé) à chaque produit
df['sujet_dominant'] = topic_encoded_df.idxmax(axis=1).str.replace('Sujet_', '').astype(int)

# Distribution des sujets dominants
topic_dist = df['sujet_dominant'].value_counts().reset_index()
topic_dist.columns = ['sujet', 'compte']
topic_dist['sujet'] = topic_dist['sujet'].astype(str)

fig = px.bar(topic_dist, x='sujet', y='compte',
             title='Distribution des Sujets Dominants dans les Produits',
             color='compte', color_continuous_scale='Purples',
             template='plotly_white', height=380)
fig.update_layout(xaxis_title='ID du Sujet', yaxis_title='Nombre de Produits')
fig.show()

### 5.2 Nuages de Mots (WordClouds) par Sujet LSA

In [ ]:
# Nuages de mots pour chaque sujet – construits à partir des mots les mieux classés pondérés par leur charge LSA
n_topics_display = min(N_TOPICS, 12)
cols_t = 3
rows_t = int(np.ceil(n_topics_display / cols_t))
fig_t, axes_t = plt.subplots(rows_t, cols_t, figsize=(18, 5 * rows_t))
axes_t = axes_t.flatten()

for topic_idx in range(n_topics_display):
    component = lsa.components_[topic_idx]
    # Utiliser les charges positives comme poids de mots pour le nuage de mots
    pos_mask = component > 0
    word_weights = {feature_names_lsa[i]: component[i]
                    for i in range(len(feature_names_lsa)) if pos_mask[i]}
    if word_weights:
        wc = WordCloud(width=400, height=250,
                       background_color='white',
                       colormap='Purples',
                       max_words=40).generate_from_frequencies(word_weights)
        axes_t[topic_idx].imshow(wc, interpolation='bilinear')
    axes_t[topic_idx].set_title(f'Sujet {topic_idx}', fontsize=12, fontweight='bold')
    axes_t[topic_idx].axis('off')

for j in range(n_topics_display, len(axes_t)):
    axes_t[j].axis('off')

plt.suptitle('Nuages de Mots – Sujets Latents LSA', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('wordclouds_topics.png', dpi=120, bbox_inches='tight')
plt.show()

### 5.3 Carte de Chaleur – Produits vs Sujets

**Pourquoi cette visualisation ?**  
Contrairement au clustering, LSA permet à chaque produit d'appartenir à **plusieurs sujets simultanément**. La carte de chaleur montre la distribution des poids de sujets sur les 50 premiers produits, validant que les sujets sont significatifs et creux.

In [ ]:
# Carte de chaleur : 50 premiers produits × tous les sujets
heatmap_data = topic_encoded_df.iloc[:50]

fig = px.imshow(
    heatmap_data.T,
    title='Poids des Sujets LSA – 50 Premiers Produits',
    labels={'x': 'Index Produit', 'y': 'Sujet', 'color': 'Poids'},
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    template='plotly_white',
    height=450
)
fig.update_layout(xaxis_title='Index Produit', yaxis_title='Sujet')
fig.show()

---
## 6. Résumé des Résultats

### Performance des Modèles

In [ ]:
print('='*65)
print('RÉSUMÉ FINAL DES PERFORMANCES DES MODÈLES')
print('='*65)

print('\n--- PARTIE 1 : Clustering DBSCAN ---')
print(f'  eps                                   : {BEST_EPS}')
print(f'  min_samples                           : 2')
print(f'  Métrique de distance                  : cosinus')
print(f'  Nombre de clusters trouvés            : {n_clusters}')
print(f'  Valeurs aberrantes                    : {n_outliers} / {len(df)} ({n_outliers/len(df)*100:.1f}%)')
if mask.sum() > 1 and n_clusters >= 2:
    print(f'  Score de Silhouette (cosinus)         : {sil_score:.4f}')
    print(f'  Interprétation                        : Séparation {"Bonne" if sil_score > 0.3 else "Modérée" if sil_score > 0.1 else "Faible"}')

print('\n--- PARTIE 2 : Système de Recommandation ---')
print(f'  Méthode      : Similarité cosinus basée sur les clusters')
print(f'  Top-N sortie : 5 recommandations par requête')
print(f'  Couverture   : {(df["cluster"] != -1).sum()} / {len(df)} produits ({(df["cluster"] != -1).sum()/len(df)*100:.1f}%)')

print('\n--- PARTIE 3 : Modélisation de Sujets LSA ---')
print(f'  Modèle                    : TruncatedSVD (LSA)')
print(f'  Nombre de sujets          : {N_TOPICS}')
print(f'  Variance expliquée        : {cumulative_variance[-1]*100:.1f}%')
print(f'  Importance des features   : Top mots par sujet (charges SVD)')
print('='*65)

---
## 7. Recommandations Métier

Sur la base de l'analyse ci-dessus :

| Constat | Action Métier |
|---------|---------------|
| Les produits se regroupent naturellement par **type de matériau** (polyester, nylon, coton bio) et **activité** (pêche, running, escalade) | Créer des filtres de catégories sur le site web alignés sur ces clusters |
| Le système de recommandation atteint une **haute similarité intra-cluster** | Déployer un widget "Vous aimerez aussi" alimenté par ce modèle |
| LSA révèle **15 sujets latents** pas toujours alignés avec les catégories existantes | Affiner les menus de navigation du site web et ajouter des tags basés sur les sujets latents |
| ~30% des produits sont des **aberrants DBSCAN** | Revoir ces produits pour un éventuel recatégorisation ou positionnement de niche |